# ?? Chicago Food Inspections ? Day 2: Data Profiling & Quality Audit

**Objective:** Build an automated data profiling baseline to understand exactly how messy the raw dataset is *before* writing cleaning transformations.

**Dataset:** City of Chicago Food Inspections (315,963 records, 17 raw features)

### 1. Setup & Environment

In [ ]:
import os
import sys
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.profiling import load_dataset, get_dataset_overview, build_column_profile

with open(os.path.join(PROJECT_ROOT, 'config', 'config.yaml'), 'r') as f:
    config = yaml.safe_load(f)

print('Configuration and modules loaded successfully.')

### 2. Load Raw Dataset & Overview Metrics

In [ ]:
raw_csv_path = os.path.join(PROJECT_ROOT, config['data']['raw_path'])
df = load_dataset(raw_csv_path)
overview = get_dataset_overview(df)

print(f"Total Rows:                 {overview['total_rows']:,}")
print(f"Total Columns:              {overview['total_columns']}")
print(f"Memory Usage:               {overview['memory_mb']:.2f} MB")
print(f"Exact Duplicate Rows:       {overview['exact_duplicate_rows']}")
print(f"Duplicate Inspection IDs:   {overview['duplicate_inspection_ids']}")
print(f"Total Missing Cells:        {overview['total_missing_cells']:,} ({overview['overall_missing_pct']}%) ")

### 3. Load Baseline Data Quality Report

We load the automatically generated baseline report created by `src/profiling.py`.

In [ ]:
report_path = os.path.join(PROJECT_ROOT, 'reports', 'data_quality_before.csv')
quality_report = pd.read_csv(report_path)
quality_report[['column_name', 'data_type', 'missing_count', 'missing_pct', 'unique_count', 'suspected_data_quality_issues']]

### 4. Missing Values Audit

In [ ]:
missing_summary = quality_report[quality_report['missing_count'] > 0][['column_name', 'missing_count', 'missing_pct']].sort_values(by='missing_pct', ascending=False)
print('Columns with Missing Data:')
print(missing_summary.to_string(index=False))

# Note: Violations has 28.19% missing data. When an establishment passes with zero violations, this field is naturally NaN.

### 5. Categorical Column Anomalies

#### 5.1 City Variations & Typos

In [ ]:
print('Top 15 Raw City Values in dataset:')
print(df['City'].value_counts(dropna=False).head(15))

# Notice typos: 'CCHICAGO', 'CHICAGOCHICAGO', 'CHICAGOO', 'CHicago', 'CHICAGO.'

#### 5.2 Facility Type Cardinality & Casing Explosion

In [ ]:
raw_types = df['Facility Type'].dropna().nunique()
lower_types = df['Facility Type'].dropna().str.lower().str.strip().nunique()
print(f"Raw Unique Facility Types:         {raw_types}")
print(f"Unique Types after Case Normalization: {lower_types}")
print(f"Difference caused by whitespace & casing: {raw_types - lower_types}")

print('\nTop 10 Facility Types:')
print(df['Facility Type'].value_counts().head(10))

#### 5.3 Risk Categories

In [ ]:
print('Risk Distribution:')
print(df['Risk'].value_counts(dropna=False))
# Notice: 83 rows contain 'All', which is not a standard Chicago food risk tier (Risk 1 High, Risk 2 Medium, Risk 3 Low).

### 6. Numerical & Datetime Anomalies

In [ ]:
print(f"License # equal to 0: {(df['License #'] == 0).sum()} rows")
print(f"Zip Codes outside Chicago (non-606xx): {len(df[~df['Zip'].fillna(0).astype(int).astype(str).str.startswith('606')])} rows")
print(f"Inspection Date is type '{df['Inspection Date'].dtype}' ? min: {df['Inspection Date'].min()} | max: {df['Inspection Date'].max()}")

### 7. Day 2 Takeaways & Pipeline Roadmap

1. **Zero Row Deletions Today:** We preserved the original 315,963 records while diagnosing root causes.
2. **Address & City Normalization Needed:** Over 246,000 address fields have trailing whitespace; City has at least 8 distinct typo variations of 'CHICAGO'.
3. **Violations Log Parser Required:** 225,170 free-text violation logs contain multi-part violation codes, descriptions, and inspector comments needing extraction.
4. **Type Casting:** License # and Zip should be cleaned strings; Inspection Date must be converted to datetime.